In [ ]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [2]:
from models import Ride, ride_deserializer

In [3]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=ride_deserializer,
    consumer_timeout_ms=100000   # auto-stops after 10s of silence
)

In [4]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [5]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
over_5_count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromisoformat(ride.lpep_pickup_datetime)
    dropoff_dt = datetime.fromisoformat(ride.lpep_dropoff_datetime)
    distance = ride.trip_distance


    # cur.execute(
    #     """INSERT INTO processed_events
    #        (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime, dropoff_datetime, passenger_count, tip_amount)
    #        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
    #     (ride.PULocationID, ride.DOLocationID,
    #      ride.trip_distance, ride.total_amount, pickup_dt, 
    #      dropoff_dt, ride.passenger_count, ride.tip_amount)
    # )
    
    if distance is not None and float(distance) > 5:
        over_5_count += 1

    count += 1
    if count % 1000 == 0:
        print(f"Inserted {count} rows... trips > 5 so far: {over_5_count}")        


consumer.close()
print("total messages:", count)
print("trip_distance > 5:", over_5_count)
cur.close()
conn.close()

Listening to green-trips-2 and writing to PostgreSQL...
Inserted 1000 rows... trips > 5 so far: 134
Inserted 2000 rows... trips > 5 so far: 288
Inserted 3000 rows... trips > 5 so far: 445
Inserted 4000 rows... trips > 5 so far: 570
Inserted 5000 rows... trips > 5 so far: 732
Inserted 6000 rows... trips > 5 so far: 912
Inserted 7000 rows... trips > 5 so far: 1036
Inserted 8000 rows... trips > 5 so far: 1167
Inserted 9000 rows... trips > 5 so far: 1301
Inserted 10000 rows... trips > 5 so far: 1443
Inserted 11000 rows... trips > 5 so far: 1564
Inserted 12000 rows... trips > 5 so far: 1740
Inserted 13000 rows... trips > 5 so far: 1840
Inserted 14000 rows... trips > 5 so far: 1954
Inserted 15000 rows... trips > 5 so far: 2106
Inserted 16000 rows... trips > 5 so far: 2268
Inserted 17000 rows... trips > 5 so far: 2375
Inserted 18000 rows... trips > 5 so far: 2483
Inserted 19000 rows... trips > 5 so far: 2622
Inserted 20000 rows... trips > 5 so far: 2744
Inserted 21000 rows... trips > 5 so far